<div align="left" style="background-color: #008080; padding: 20px 10px;">
<h3><b style="color: white;">IDEAS - Institute of Data Engineering, Analytics and Science Foundation</b></h3>
<p style="color: white;">Autumn Internship Program 2026</p>
<hr style="width:75%; background-color: white; border-color: white;">
<h3 style="color: white;">Project Title: Industrial Product Demand Forecasting</h3>
<h4 style="color: white;">Mentor Name: Dr. Chandan Biswas</h4>
</div>

### Question 1: Data Loading & Missing Value Justification (5 Marks)

Download the 'Historical Product Demand' dataset from Kaggle (`https://www.kaggle.com/datasets/felixzhao/productdemandforecasting`). Load the CSV file into a pandas DataFrame. Parse the 'Date' column as a datetime object. Identify missing values in the Date column and handle them. Print your written answer using Python `print()` statements to logically justify your chosen method for handling missing dates in a time-series context.

**Hint:** Use `pd.read_csv()` and `pd.to_datetime()`. Assess missing values. Use `print()` to state why dropping or imputing missing dates makes logical sense before forecasting.

**Expected Output:** A pandas DataFrame with cleaned data and a printed analytical justification.

In [22]:
import pandas as pd

# 1. Load the dataset
file_path = 'Historical Product Demand.csv'
df_q1 = pd.read_csv(file_path)

# 2. Parse the 'Date' column as a datetime object
df_q1['Date'] = pd.to_datetime(df_q1['Date'], errors='coerce')

# 3. Assess and handle missing values in 'Date'
initial_rows = len(df_q1)
missing_dates_count = df_q1['Date'].isnull().sum()

# Drop rows where 'Date' is null
df_q1_cleaned = df_q1.dropna(subset=['Date']).copy()
final_rows = len(df_q1_cleaned)

# Display verification
print(f"Initial number of records: {initial_rows}")
print(f"Number of missing dates identified and removed: {missing_dates_count}")
print(f"Number of records after cleaning: {final_rows}\n")


Initial number of records: 1048575
Number of missing dates identified and removed: 11239
Number of records after cleaning: 1037336



### Question 2: Target Variable Cleaning (5 Marks)

The target variable 'Order_Demand' contains non-numeric string characters (such as parentheses) in some rows. Remove all non-numeric characters from the 'Order_Demand' column and convert the column to an integer data type.

**Hint:** You can use the `.str.replace()` method with a regular expression (like `[^0-9]`) to strip out anything that is not a digit, followed by `.astype(int)`.

**Expected Output:** A pandas DataFrame where the 'Order_Demand' column is structured strictly as an integer data type.

In [23]:
# Remove all non-numeric characters from 'Order_Demand'
df_q1_cleaned['Order_Demand'] = df_q1_cleaned['Order_Demand'].astype(str).str.replace(r'[^0-9]', '', regex=True)

# Convert the column to integer
df_q1_cleaned['Order_Demand'] = df_q1_cleaned['Order_Demand'].astype(int)

# Confirm the data type and display sample results
print("Data type of 'Order_Demand':", df_q1_cleaned['Order_Demand'].dtype)
display(df_q1_cleaned[['Product_Code', 'Order_Demand']].head())

Data type of 'Order_Demand': int64


,Product_Code,Order_Demand
0,Product_0993,100
1,Product_0979,500
2,Product_0979,500
3,Product_0979,500
4,Product_0979,500


### Question 3: Time-Series Aggregation (6 Marks)

Group the DataFrame by 'Date' and calculate the sum of 'Order_Demand' to simplify the multi-product dataset into a single temporal sequence. Store the result in a new DataFrame where the index represents the Date and a single column represents the aggregated daily demand.

**Hint:** Use the `.groupby()` method on the 'Date' column and apply `.sum()` to the 'Order_Demand' column. Ensure the result is a DataFrame rather than a pandas Series.

**Expected Output:** A time-series DataFrame indexed chronologically by date, containing a single numeric column for aggregate demand.

In [24]:
# Group by 'Date' and sum 'Order_Demand' to create a single chronological time series sequence
df_daily_demand = df_q1_cleaned.groupby('Date')[['Order_Demand']].sum()

# Sort the index chronologically to guarantee correct time series alignment
df_daily_demand = df_daily_demand.sort_index()

# Confirm properties of the new aggregated DataFrame
print("Index type:", df_daily_demand.index.dtype)
print("Shape of aggregated DataFrame:", df_daily_demand.shape)
display(df_daily_demand.head())

Index type: datetime64[ns]
Shape of aggregated DataFrame: (1729, 1)


,Order_Demand
Date,
2011-01-08,2
2011-05-31,108
2011-06-24,92000
2011-09-02,1250
2011-09-27,28


### Question 4: Advanced Time-Series Feature Engineering (8 Marks)

Generate chronological and statistical features from the aggregated daily demand DataFrame. Extract 'Year', 'Month', and 'DayOfWeek' from the datetime index. Additionally, create a 7-day lag feature ('Lag_7') and a 7-day rolling mean ('Rolling_Mean_7'). Drop any rows containing missing values resulting from these operations.

**Hint:** Use the `.index.year`, `.index.month`, and `.index.dayofweek` attributes. For statistical features, use `.shift(7)` and `.rolling(window=7).mean()`. Follow up with `.dropna()`.

**Expected Output:** A DataFrame expanded with multiple newly engineered chronological and statistical numerical features.

In [25]:
# Create a copy of the daily demand DataFrame to keep operations safe
df_features = df_daily_demand.copy()

# 1. Extract chronological features from the index
df_features['Year'] = df_features.index.year
df_features['Month'] = df_features.index.month
df_features['DayOfWeek'] = df_features.index.dayofweek

# 2. Create statistical features: 7-day lag and 7-day rolling mean
df_features['Lag_7'] = df_features['Order_Demand'].shift(7)
df_features['Rolling_Mean_7'] = df_features['Order_Demand'].rolling(window=7).mean()

# Record size before dropping NaNs
rows_before_drop = len(df_features)

# 3. Drop any rows with missing values resulting from these lag and window operations
df_features_cleaned = df_features.dropna()

# Confirm properties and display results
print(f"Rows before dropping NaNs: {rows_before_drop}")
print(f"Rows after dropping NaNs (first 7 rows lost due to rolling window): {len(df_features_cleaned)}")
display(df_features_cleaned.head(10))

Rows before dropping NaNs: 1729
Rows after dropping NaNs (first 7 rows lost due to rolling window): 1722


,Order_Demand,Year,Month,DayOfWeek,Lag_7,Rolling_Mean_7
Date,,,,,,
2011-10-31,3,2011,10,0,2.0,14120.428571
2011-11-04,500,2011,11,4,108.0,14176.428571
2011-11-18,4162,2011,11,4,92000.0,1628.142857
2011-11-21,110,2011,11,0,1250.0,1465.285714
2011-11-22,5740,2011,11,1,28.0,2281.285714
2011-11-23,1200,2011,11,2,5450.0,1674.142857
2011-11-24,42500,2011,11,3,4.0,7745.000000
2011-11-25,17250,2011,11,4,3.0,10208.857143
2011-11-28,310,2011,11,0,500.0,10181.714286


### Question 5: Outlier Detection and Analytical Reasoning (6 Marks)

Identify and remove outliers using the Z-score method. Calculate the Z-score for the daily demand feature. Filter the DataFrame to retain rows where the absolute Z-score is less than or equal to 3.0. Print your written answer using Python `print()` statements to explain the risk of retaining extreme outliers in a forecasting model.

**Hint:** Utilize `scipy.stats.zscore` or compute it manually. Apply boolean indexing to filter the rows. Output your reasoning via the print function.

**Expected Output:** A filtered DataFrame with anomaly days removed and a printed statement explaining the mathematical impact of outliers on machine learning algorithms.

In [26]:
from scipy.stats import zscore

# Create a copy of clean feature DataFrame
df_z = df_features_cleaned.copy()

# Calculate the Z-score for 'Order_Demand'
df_z['Zscore_Demand'] = zscore(df_z['Order_Demand'])

# Filter to retain rows where the absolute Z-score is less than or equal to 3.0
outlier_filtered = df_z[df_z['Zscore_Demand'].abs() <= 3.0].drop(columns=['Zscore_Demand']).copy()

# Print properties
print(f"Rows before outlier filtering: {len(df_z)}")
print(f"Rows after outlier filtering: {len(outlier_filtered)}")
print(f"Number of outlier rows removed: {len(df_z) - len(outlier_filtered)}\n")

# Display sample of cleaned data
display(outlier_filtered.head())


Rows before outlier filtering: 1722
Rows after outlier filtering: 1718
Number of outlier rows removed: 4



,Order_Demand,Year,Month,DayOfWeek,Lag_7,Rolling_Mean_7
Date,,,,,,
2011-10-31,3,2011,10,0,2.0,14120.428571
2011-11-04,500,2011,11,4,108.0,14176.428571
2011-11-18,4162,2011,11,4,92000.0,1628.142857
2011-11-21,110,2011,11,0,1250.0,1465.285714
2011-11-22,5740,2011,11,1,28.0,2281.285714


### Question 6: Exploratory Data Analysis and Correlation (7 Marks)

Compute the Pearson correlation matrix for all features and the target variable. Extract the correlation values specifically related to the demand target. Print your written answer using Python `print()` statements to identify the most correlated engineered feature and explain why this logical relationship exists.

**Hint:** Use the `.corr()` method on the DataFrame. Select the column corresponding to the target demand. Use `print()` to analyze the strongest temporal relationship.

**Expected Output:** A pandas Series showing correlation coefficients relative to the target and a printed analytical interpretation.

In [27]:
# Compute Pearson correlation with the target 'Order_Demand' for all engineered features
correlation = outlier_filtered.corr(numeric_only=True)["Order_Demand"].sort_values(ascending=False)
print("=== Pearson Correlation with Target 'Order_Demand' ===")
print(correlation)


=== Pearson Correlation with Target 'Order_Demand' ===
Order_Demand      1.000000
Rolling_Mean_7    0.416074
Lag_7             0.355340
Year              0.110659
Month            -0.043799
DayOfWeek        -0.611881
Name: Order_Demand, dtype: float64


### Question 7: Chronological Train-Test Split (5 Marks)

Split the dataset into training and testing sets while preserving the temporal order. Allocate the first 80% of the rows to the training set and the remaining 20% to the testing set. Separate the features into a matrix `X` and the target into a vector `y` for both subsets.

**Hint:** Avoid random splitting (do not use default `train_test_split` shuffling). Calculate the integer index corresponding to the 80% threshold and slice the DataFrames chronologically.

**Expected Output:** Four distinct variables containing the training features, testing features, training target, and testing target, strictly preserving temporal sequence.

In [28]:
model_df = outlier_filtered.copy()

# Separate the featuresX and targety
X = model_df.drop(columns=["Order_Demand"])
y = model_df["Order_Demand"]

# Calculate the integer with 80% threshold
split_idx = int(len(model_df) * 0.8)

# Slice the DataFrames chronologically
X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

# Output properties to confirm the correct split
print(f"Total sequence length: {len(model_df)}")
print(f"Training set size (80%): {len(X_train)}")
print(f"Testing set size (20%): {len(X_test)}")
print("\nFirst few rows of Training features:")
display(X_train.head())
print("\nFirst few rows of Testing features:")
display(X_test.head())

Total sequence length: 1718
Training set size (80%): 1374
Testing set size (20%): 344

First few rows of Training features:


,Year,Month,DayOfWeek,Lag_7,Rolling_Mean_7
Date,,,,,
2011-10-31,2011,10,0,2.0,14120.428571
2011-11-04,2011,11,4,108.0,14176.428571
2011-11-18,2011,11,4,92000.0,1628.142857
2011-11-21,2011,11,0,1250.0,1465.285714
2011-11-22,2011,11,1,28.0,2281.285714



First few rows of Testing features:


,Year,Month,DayOfWeek,Lag_7,Rolling_Mean_7
Date,,,,,
2015-12-13,2015,12,6,100000.0,3.751713e+06
2015-12-14,2015,12,0,1097911.0,4.063693e+06
2015-12-15,2015,12,1,5747132.0,3.966204e+06
2015-12-16,2015,12,2,4642707.0,3.649865e+06
2015-12-17,2015,12,3,4977944.0,3.392705e+06


### Question 8: Feature Scaling and DataFrame Restoration (6 Marks)

Initialize a `StandardScaler`. Fit the scaler exclusively on the training features `X_train` and apply the transformation to both `X_train` and `X_test`. Explicitly convert the resulting scaled NumPy arrays back into pandas DataFrames, retaining the original column names and indices.

**Hint:** Import `StandardScaler`. Use `.fit_transform()` on training and `.transform()` on testing data. Reconstruct the DataFrame using `pd.DataFrame(data=..., index=..., columns=...)`.

**Expected Output:** Two pandas DataFrames containing standardized feature values, properly labeled with their original column headers.

In [29]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Initialize the StandardScaler
scaler = StandardScaler()

# Fit exclusively on X_train and transform both splits
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Restore into pandas DataFrames, retaining original indices and column names
X_train_scaled_df = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)

# Confirm properties and display samples
print(f"Scaled Train Shape: {X_train_scaled_df.shape}")
print(f"Scaled Test Shape: {X_test_scaled_df.shape}\n")

print("First 5 rows of Scaled Training Features:")
display(X_train_scaled_df.head())

print("\nFirst 5 rows of Scaled Testing Features:")
display(X_test_scaled_df.head())

Scaled Train Shape: (1374, 5)
Scaled Test Shape: (344, 5)

First 5 rows of Scaled Training Features:


,Year,Month,DayOfWeek,Lag_7,Rolling_Mean_7
Date,,,,,
2011-10-31,-2.08349,0.98172,-1.430360,-1.441343,-3.850052
2011-11-04,-2.08349,1.26750,0.584831,-1.441291,-3.849978
2011-11-18,-2.08349,1.26750,0.584831,-1.396485,-3.866381
2011-11-21,-2.08349,1.26750,-1.430360,-1.440734,-3.866594
2011-11-22,-2.08349,1.26750,-0.926562,-1.441330,-3.865527



First 5 rows of Scaled Testing Features:


,Year,Month,DayOfWeek,Lag_7,Rolling_Mean_7
Date,,,,,
2015-12-13,1.379123,1.55328,1.592426,-1.392584,1.035506
2015-12-14,1.379123,1.55328,-1.430360,-0.906007,1.443307
2015-12-15,1.379123,1.55328,-0.926562,1.360937,1.315876
2015-12-16,1.379123,1.55328,-0.422765,0.822423,0.902377
2015-12-17,1.379123,1.55328,0.081033,0.985883,0.566232


### Question 9: Unsupervised Learning for Demand Regimes (7 Marks)

Perform unsupervised clustering to identify latent demand states. Initialize a K-Means algorithm with 3 clusters. Fit the algorithm on the scaled training features. Predict the cluster labels for both the training and testing sets, and append these labels as a new column named 'Demand_Regime' to the respective DataFrames.

**Hint:** Use `KMeans` from `sklearn.cluster`. After fitting, use the `.predict()` method. Assign the output arrays directly as a new column in your scaled training and testing DataFrames.

**Expected Output:** The training and testing DataFrames updated with a new categorical feature representing discrete cluster assignments.

In [30]:
from sklearn.cluster import KMeans

# Initialize K-Means clustering algorithm with 3 clusters
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)

# Fit on scaled training features and predict for both training and testing sets
X_train_scaled_df["Demand_Regime"] = kmeans.fit_predict(X_train_scaled_df)

X_test_scaled_df["Demand_Regime"] = kmeans.predict(X_test_scaled_df)

# Confirm new column addition and display distribution of clusters
print("Train Set Cluster Distribution:")
print(X_train_scaled_df["Demand_Regime"].value_counts())

print("\nTest Set Cluster Distribution:")
print(X_test_scaled_df["Demand_Regime"].value_counts())

print("\nFirst 5 rows of Updated Scaled Training DataFrame:")
display(X_train_scaled_df.head())

Train Set Cluster Distribution:
Demand_Regime
1    538
2    535
0    301
Name: count, dtype: int64

Test Set Cluster Distribution:
Demand_Regime
1    306
0     38
Name: count, dtype: int64

First 5 rows of Updated Scaled Training DataFrame:


,Year,Month,DayOfWeek,Lag_7,Rolling_Mean_7,Demand_Regime
Date,,,,,,
2011-10-31,-2.08349,0.98172,-1.430360,-1.441343,-3.850052,2
2011-11-04,-2.08349,1.26750,0.584831,-1.441291,-3.849978,2
2011-11-18,-2.08349,1.26750,0.584831,-1.396485,-3.866381,2
2011-11-21,-2.08349,1.26750,-1.430360,-1.440734,-3.866594,2
2011-11-22,-2.08349,1.26750,-0.926562,-1.441330,-3.865527,2


### Question 10: Baseline Supervised Model and Cross-Validation (8 Marks)

Instantiate a `RandomForestRegressor` with a fixed random state. Perform 5-fold Time Series cross-validation on the training set to compute the average Mean Absolute Error (MAE). Print the resulting average MAE.

**Hint:** Import `TimeSeriesSplit` and `cross_val_score` from `sklearn.model_selection`. Set the scoring parameter to `'neg_mean_absolute_error'`, convert the array to positive values, and calculate the mean.

**Expected Output:** A printed numerical value representing the average cross-validated Mean Absolute Error for the baseline model.

In [31]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

# Instantiate baseline model
rf_model = RandomForestRegressor(random_state=42)

# Initialize 
time_series_cv = TimeSeriesSplit(n_splits=5)

# Compute cross-validated scores on the scaled training features and training targets
cv_scores = cross_val_score(
    rf_model,
    X_train_scaled_df,
    y_train,
    cv=time_series_cv,
    scoring="neg_mean_absolute_error"
)

# Calculate and print the positive average MAE
avg_mae = -cv_scores.mean()
print("=== Baseline Supervised Model Evaluation + Cross-Validation ===")
print("Average 5-Fold Time Series Cross-Validated MAE:", avg_mae)

=== Baseline Supervised Model Evaluation + Cross-Validation ===
Average 5-Fold Time Series Cross-Validated MAE: 819899.8389519651


### Question 11: Hyperparameter Tuning (8 Marks)

Optimize the `RandomForestRegressor` using `GridSearchCV` combined with `TimeSeriesSplit`. Define a parameter grid searching across `n_estimators` (50, 100) and `max_depth` (None, 10). Fit the grid search on the training data and extract the optimal model configuration.

**Hint:** Define a dictionary mapping parameters to lists of values. Pass the model, grid, and cross-validator to `GridSearchCV`. Access the best hyperparameters using `.best_params_`.

**Expected Output:** An executed cell that tunes the model over the specified parameter space and prints the optimal hyperparameters.

In [32]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

# Define the parameter grid to search over
param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [None, 10]
}

# Set up the TimeSeriesSplit cross-validator
time_series_cv = TimeSeriesSplit(n_splits=5)

# Initialize GridSearchCV
rf_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    cv=time_series_cv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

# Fit the grid search on the scaled training features and training targets
rf_search.fit(X_train_scaled_df, y_train)

# Output the optimal model parameters and corresponding MAE score
print("=== Hyperparameter Tuning Results ===")
print("Best Hyperparameters:", rf_search.best_params_)
print("Best Cross-Validated MAE:", -rf_search.best_score_)

=== Hyperparameter Tuning Results ===
Best Hyperparameters: {'max_depth': 10, 'n_estimators': 50}
Best Cross-Validated MAE: 803109.2930834221


### Question 12: Tuned Model Evaluation and Analytical Reasoning (7 Marks)

Evaluate the optimized Random Forest model by generating predictions on the scaled testing features. Calculate the Mean Absolute Error (MAE) and the Root Mean Squared Error (RMSE). Print your written answer using Python `print()` statements to compare these test metrics against the baseline cross-validation score and determine if hyperparameter tuning yielded significant improvements.

**Hint:** Use `mean_absolute_error` and `mean_squared_error` (compute the square root). Structure a text response analyzing the magnitude of improvement and output it via `print()`.

**Expected Output:** Calculated error metrics and a printed written analysis objectively evaluating the impact of the tuning phase.

In [33]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# 1. Get the best estimator from our grid search
best_model = rf_search.best_estimator_

# 2. Predict on testing features
y_pred = best_model.predict(X_test_scaled_df)

# 3. Calculate 
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# 4. Print
print("=== Model Evaluation Metrics ===")
print(f"Baseline Cross-Validated MAE : {avg_mae:,.2f}")
print(f"Tuned Test Set MAE            : {mae:,.2f}")
print(f"Tuned Test Set RMSE           : {rmse:,.2f}\n")

=== Model Evaluation Metrics ===
Baseline Cross-Validated MAE : 819,899.84
Tuned Test Set MAE            : 932,061.20
Tuned Test Set RMSE           : 1,283,625.30



### Question 13: PyTorch Data Preparation and Skeleton Architecture (8 Marks)

Convert the scaled `X_train`, `X_test`, `y_train`, and `y_test` DataFrames/Series into PyTorch Tensors with a `torch.float32` data type. Reshape the target tensors into column vectors. Then, utilize the provided skeleton code to construct a Multi-Layer Perceptron (MLP) containing one hidden layer with 64 neurons, a ReLU activation function, and a final output layer. Fill in the missing architectural parameters.

**Hint:** Complete the `.tensor()` instantiations. Use `.unsqueeze(1)` or `.view(-1, 1)` for the targets. In the `MLP` class, fill in the `in_features` and `out_features` mapping to construct the network dimensions accurately.

**Expected Output:** Four correctly shaped PyTorch tensor objects and an instantiated object representing the completed MLP architecture.

In [34]:
import torch
import torch.nn as nn

# 1. Convert DataFrames into float32 type
X_train_tensor = torch.tensor(X_train_scaled_df.values, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled_df.values, dtype=torch.float32)

# 2. Convert targets and reshape into column vectors (.view(-1, 1))
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.float32).view(-1, 1)

# 3. Define the (MLP) model structure
class MLP(nn.Module):
    def __init__(self, input_size):
        super(MLP, self).__init__()
        # Hidden layer 64 neurons
        self.hidden = nn.Linear(in_features=input_size, out_features=64)
        # ReLU activation
        self.relu = nn.ReLU()
        # Output layer
        self.output = nn.Linear(in_features=64, out_features=1)

    def forward(self, x):
        x = self.hidden(x)
        x = self.relu(x)
        x = self.output(x)
        return x

# 4. Instantiation
input_size = X_train_tensor.shape[1]
model = MLP(input_size=input_size)

# 5. Display 
print("=== PyTorch Data Preparation & Architecture ===")
print("X_train_tensor shape :", X_train_tensor.shape)
print("y_train_tensor shape :", y_train_tensor.shape)
print("X_test_tensor shape  :", X_test_tensor.shape)
print("y_test_tensor shape  :", y_test_tensor.shape)
print("\nModel Architecture:")
print(model)

=== PyTorch Data Preparation & Architecture ===
X_train_tensor shape : torch.Size([1374, 6])
y_train_tensor shape : torch.Size([1374, 1])
X_test_tensor shape  : torch.Size([344, 6])
y_test_tensor shape  : torch.Size([344, 1])

Model Architecture:
MLP(
  (hidden): Linear(in_features=6, out_features=64, bias=True)
  (relu): ReLU()
  (output): Linear(in_features=64, out_features=1, bias=True)
)


### Question 14: Multi-Layer Perceptron Training Loop (8 Marks)

Instantiate the MLP model. Define a Mean Squared Error (MSE) loss function and an Adam optimizer (learning rate = 0.001). Construct a training loop iterating over 100 epochs. Within each epoch, compute the forward pass, calculate the loss, execute a backward pass to compute gradients, and update the model weights. Print the loss every 20 epochs.

**Hint:** The standard order of operations inside the loop is: `optimizer.zero_grad()`, `model(inputs)`, `criterion(outputs, targets)`, `loss.backward()`, and `optimizer.step()`.

**Expected Output:** A printed log tracking the numerical reduction of the loss function over consecutive epochs.

In [35]:
import torch
import torch.nn as nn

# 1. Instantiate the network model explicitly using our prepared input dimension
input_size = X_train_tensor.shape[1]
model = MLP(input_size=input_size)

# 2. Define the loss function (MSE) and optimizer (Adam with learning rate 0.001)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("=== Starting Neural Network Training =+=")
# 3. Training loop iteration over 100 epochs
for epoch in range(100):
    optimizer.zero_grad()
    
    
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()
    
    # Output loss tracking every 20 epochs
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch + 1:3d}/100 | Training MSE Loss: {loss.item():,.2f}")

=== Starting Neural Network Training =+=
Epoch  20/100 | Training MSE Loss: 12,788,895,318,016.00
Epoch  40/100 | Training MSE Loss: 12,788,893,220,864.00
Epoch  60/100 | Training MSE Loss: 12,788,889,026,560.00
Epoch  80/100 | Training MSE Loss: 12,788,883,783,680.00
Epoch 100/100 | Training MSE Loss: 12,788,877,492,224.00


### Question 15: Business Impact Evaluation (6 Marks)

Evaluate the MLP predictions on the test set using a custom business penalty metric. In this supply chain scenario, under-predicting demand incurs a penalty equal to 2.0 times the absolute error, while over-predicting incurs a penalty of 1.0 times the absolute error. Calculate the total aggregated penalty. Print your written answer using Python `print()` statements to explain why under-prediction is typically penalized more severely in inventory management.

**Hint:** Calculate raw errors (`actual - predicted`). Apply conditional logic (e.g., boolean masks) to separate negative and positive errors, multiply by the specified weights, and compute the sum. Include the business justification via print.

**Expected Output:** A numerical calculation representing the simulated business cost of the forecasting errors and a printed logical justification for asymmetric penalties.

In [36]:
import numpy as np
import torch

# 1. Generate model predictions 
with torch.no_grad():
    y_pred_tensor = model(X_test_tensor)

# 2. Convert PyTorch tensors back to flat NumPy array
y_pred = y_pred_tensor.numpy().ravel()
y_actual = y_test_tensor.numpy().ravel()

# 3. Calculate raw error (Actual - Predicted)
errors = y_actual - y_pred

# 4. Separate under-predictions (Actual > Predicted) and over-predictions (Actual <= Predicted)
under_prediction_mask = errors > 0
over_prediction_mask = errors <= 0

# 5. Apply the asymmetric business weights (Under-predict penalty: 2.0, Over-predict penalty: 1.0)
penalty = (
    (under_prediction_mask * errors * 2.0) +
    (over_prediction_mask * np.abs(errors) * 1.0)
).sum()

# 6. Display numerical o/p
print("=== Supply Chain Custom Business Impact Evaluation ===")
print(f"Total Aggregated Business Penalty: {penalty:,.2f}\n")


=== Supply Chain Custom Business Impact Evaluation ===
Total Aggregated Business Penalty: 2,079,217,408.00

